# Logistic Regression CPCV

Simple CPCV workflow for L2, L1, and Elastic Net logistic regression. It uses clean feature sets, saved triple-barrier labels, purging by `timeout_date`, and an embargo equal to each label file's `num_days`.

In [30]:
from __future__ import annotations

import itertools
import os
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib

if not os.environ.get("JPY_PARENT_PID"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 100)

RUN_FULL_GRID = False
SAVE_OUTPUTS = False
RANDOM_STATE = 42
PROBA_THRESHOLD = 0.50
TRAIN_END_DATE = pd.Timestamp("2022-01-01")

SMOKE_INSTRUMENTS = ["cl1s"]
FULL_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]
SMOKE_CONFIG_NAMES = ["ewma_10d_tp2_sl2"]

INSTRUMENTS = FULL_INSTRUMENTS if RUN_FULL_GRID else SMOKE_INSTRUMENTS
C_VALUES = [0.01, 0.1, 1.0, 10.0] if RUN_FULL_GRID else [0.1, 1.0]
L1_RATIOS = [0.25, 0.5, 0.75] if RUN_FULL_GRID else [0.5]

CPCV_SETTINGS = {
    "cl1s": {"n_groups": 6, "k_test_groups": 2},
    "rb1s": {"n_groups": 6, "k_test_groups": 2},
    "ho1s": {"n_groups": 4, "k_test_groups": 1},
    "ng1s": {"n_groups": 4, "k_test_groups": 1},
}

LEAKAGE_COLUMNS = {
    "training_end", "vol", "tp", "sl", "timeout_date", "timeout_close",
    "touch_date", "touch_price", "touched_barrier", "triple_barrier_label", "metalabel",
    "volatility_method", "ewma_span", "volatility_window", "num_days", "take_profit_mult",
    "stop_loss_mult", "holding_period_days", "raw_touch_return", "signed_touch_return",
}

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "features").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with data/features")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CLEAN_DIR = PROJECT_ROOT / "data" / "features" / "clean_feature_set"
TB_DIR = PROJECT_ROOT / "data" / "features" / "triple_barrier"
TB_SUMMARY_PATH = TB_DIR / "triple_barrier_config_summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "models" / "logistic_cpcv"

print("Run full grid:", RUN_FULL_GRID)
print("Instruments:", INSTRUMENTS)
print("C values:", C_VALUES)
print("Elastic Net l1 ratios:", L1_RATIOS)

Run full grid: False
Instruments: ['cl1s']
C values: [0.1, 1.0]
Elastic Net l1 ratios: [0.5]


## 1. Load clean features and label files

In smoke mode this loads one instrument and one triple-barrier config. Set `RUN_FULL_GRID = True` above to run all four energy clean feature sets and all matching saved label files.

In [31]:
clean_features = {}

for instrument in INSTRUMENTS:
    path = CLEAN_DIR / f"{instrument}_clean_feature_set.csv"
    assert path.exists(), f"Missing clean feature file: {path}"

    df = pd.read_csv(path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    assert df["instrument"].eq(instrument).all()
    assert df["date"].duplicated().sum() == 0

    model_feature_cols = [c for c in df.columns if c not in {"date", "instrument", "primary_signal"}]
    assert df[model_feature_cols].isna().sum().sum() == 0, f"Missing feature values remain in {path}"
    clean_features[instrument] = df.drop(columns=["primary_signal"], errors="ignore")

label_files = pd.read_csv(TB_SUMMARY_PATH)
label_files = label_files[label_files["instrument"].isin(INSTRUMENTS)].copy()

if not RUN_FULL_GRID:
    label_files = label_files[label_files["config_name"].isin(SMOKE_CONFIG_NAMES)].copy()

label_files["label_path"] = label_files["output_path"].map(lambda p: str(Path(p) if Path(p).exists() else TB_DIR / Path(p).name))
missing = [p for p in label_files["label_path"] if not Path(p).exists()]
assert not missing, f"Missing label files: {missing[:5]}"
assert not label_files.empty, "No label files selected."

metalabel_distribution = label_files.assign(
    metalabel_0_count=label_files["metalabel_0_count"].astype(int),
    metalabel_1_count=label_files["metalabel_1_count"].astype(int),
    metalabel_0_rate=label_files["metalabel_0_count"] / label_files["rows"],
    metalabel_1_rate=label_files["metalabel_1_count"] / label_files["rows"],
)

display(pd.DataFrame({"instrument": k, "clean_rows": len(v), "clean_columns": v.shape[1]} for k, v in clean_features.items()))
display(
    metalabel_distribution[
        [
            "instrument", "config_name", "rows", "metalabel_0_count", "metalabel_1_count",
            "metalabel_0_rate", "metalabel_1_rate", "num_days", "label_path",
        ]
    ].head(20)
)

,instrument,clean_rows,clean_columns
0,cl1s,626,146


,instrument,config_name,rows,metalabel_0_count,metalabel_1_count,metalabel_0_rate,metalabel_1_rate,num_days,label_path
0,cl1s,ewma_10d_tp2_sl2,324,201,123,0.62037,0.37963,10,/Users/faisal/Desktop/systematic-strategies-wi...


## 2. Build modelling data and CPCV splits

The split logic has two protections:

- **Purge:** remove train events whose `date -> timeout_date` window overlaps the test event window.
- **Embargo:** remove train events immediately after the test block. Here the embargo equals `num_days` from that triple-barrier file.

In [32]:
def read_labels(label_path: str) -> pd.DataFrame:
    columns = pd.read_csv(label_path, nrows=0).columns
    date_columns = [c for c in ["date", "training_end", "timeout_date", "touch_date"] if c in columns]
    return pd.read_csv(label_path, parse_dates=date_columns).sort_values("date").reset_index(drop=True)


def make_model_data(label_row: pd.Series):
    labels = read_labels(label_row["label_path"])
    features = clean_features[label_row["instrument"]]

    labels = labels.drop(columns=["close"], errors="ignore")
    data = labels.merge(features, on=["date", "instrument"], how="inner")
    data = data.sort_values("date").reset_index(drop=True)
    data = data[data["date"] < TRAIN_END_DATE].copy()
    assert "primary_signal" in data.columns, "primary_signal missing after merge."
    data = data[(data["metalabel"].isin([0, 1])) & (data["primary_signal"] != 0)].copy()
    data = data.sort_values("date").reset_index(drop=True)

    assert not data.empty, f"No merged rows for {label_row['instrument']} {label_row['config_name']}"
    assert data["metalabel"].nunique() == 2, "Need both metalabel classes."
    assert data["num_days"].nunique() == 1, "One label file should have one num_days value."

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in LEAKAGE_COLUMNS]
    assert not set(feature_cols).intersection(LEAKAGE_COLUMNS)
    assert feature_cols, "No numeric features available."

    data[feature_cols] = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    embargo_observations = int(data["num_days"].iloc[0])
    return data, feature_cols, embargo_observations


def date_after_n_observations(dates: pd.Series, end_date: pd.Timestamp, n: int) -> pd.Timestamp:
    unique_dates = pd.Series(pd.to_datetime(dates).sort_values().unique())
    first_after_end = unique_dates.searchsorted(pd.Timestamp(end_date), side="right")
    if first_after_end >= len(unique_dates):
        return pd.Timestamp(end_date)
    return pd.Timestamp(unique_dates.iloc[min(len(unique_dates) - 1, first_after_end + n - 1)])


def make_cpcv_splits(data: pd.DataFrame, instrument: str, embargo_observations: int) -> list[dict]:
    settings = CPCV_SETTINGS[instrument]
    groups = [g.astype(int) for g in np.array_split(np.arange(len(data)), settings["n_groups"]) if len(g) > 0]
    splits = []

    for split_id, test_group_ids in enumerate(itertools.combinations(range(len(groups)), settings["k_test_groups"])):
        test_idx = np.concatenate([groups[group_id] for group_id in test_group_ids])
        train_mask = np.ones(len(data), dtype=bool)
        train_mask[test_idx] = False

        for group_id in test_group_ids:
            group_idx = groups[group_id]
            test_start = data.loc[group_idx, "date"].min()
            test_end = data.loc[group_idx, "date"].max()
            test_horizon_end = data.loc[group_idx, "timeout_date"].max()
            embargo_end = date_after_n_observations(data["date"], test_end, embargo_observations)

            overlaps_test_window = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
            inside_embargo = (data["date"] > test_end) & (data["date"] <= embargo_end)
            train_mask[overlaps_test_window | inside_embargo] = False

        train_idx = np.where(train_mask)[0]
        assert set(train_idx).isdisjoint(set(test_idx)), "Train/test overlap found."

        if len(train_idx) > 0:
            splits.append({
                "split_id": split_id,
                "instrument": instrument,
                "test_groups": test_group_ids,
                "train_idx": train_idx,
                "test_idx": test_idx,
                "train_rows": len(train_idx),
                "test_rows": len(test_idx),
                "embargo_observations": embargo_observations,
            })

    return splits


def split_status_frame(data: pd.DataFrame, split: dict) -> pd.DataFrame:
    status = pd.Series("train", index=data.index)
    status.iloc[split["test_idx"]] = "test"

    for group_id in split["test_groups"]:
        settings = CPCV_SETTINGS[split["instrument"]]
        groups = [g.astype(int) for g in np.array_split(np.arange(len(data)), settings["n_groups"]) if len(g) > 0]
        group_idx = groups[group_id]

        test_start = data.loc[group_idx, "date"].min()
        test_end = data.loc[group_idx, "date"].max()
        test_horizon_end = data.loc[group_idx, "timeout_date"].max()
        embargo_end = date_after_n_observations(data["date"], test_end, split["embargo_observations"])

        purged = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
        embargoed = (data["date"] > test_end) & (data["date"] <= embargo_end)

        status.loc[purged & status.ne("test")] = "purged"
        status.loc[embargoed & status.ne("test")] = "embargo"

    status.iloc[split["train_idx"]] = "train"
    return pd.DataFrame({"date": data["date"], "split_id": split["split_id"], "status": status.values})


def plot_cpcv_splits(data: pd.DataFrame, splits: list[dict]) -> None:
    plot_df = pd.concat([split_status_frame(data, split) for split in splits], ignore_index=True)
    colours = {"train": "lightgrey", "purged": "#d62728", "embargo": "#ff7f0e", "test": "#1f77b4"}

    plt.figure(figsize=(14, 6))
    for status_name, status_df in plot_df.groupby("status"):
        plt.scatter(
            status_df["date"],
            status_df["split_id"],
            s=18,
            marker="s",
            color=colours[status_name],
            label=status_name,
            alpha=0.9,
        )

    plt.title("CPCV splits with purging and num_days embargo")
    plt.xlabel("Event date")
    plt.ylabel("Split id")
    plt.yticks(sorted(plot_df["split_id"].unique()))
    plt.legend(loc="upper right")
    plt.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    if os.environ.get("JPY_PARENT_PID"):
        plt.show()
    else:
        plt.close()


example_data, example_features, example_embargo = make_model_data(label_files.iloc[0])
example_splits = make_cpcv_splits(example_data, label_files.iloc[0]["instrument"], example_embargo)
print("Example rows:", len(example_data))
print("Example features:", len(example_features))
print("Example embargo observations:", example_embargo)
print("Example CPCV splits:", len(example_splits))
plot_cpcv_splits(example_data, example_splits)

Example rows: 323
Example features: 145
Example embargo observations: 10
Example CPCV splits: 15


## 3. Define logistic models

In [26]:
model_specs = []

for c_value in C_VALUES:
    model_specs.append({
        "model_name": f"l2_C{c_value:g}",
        "model_family": "l2_logistic",
        "penalty": "l2",
        "C": c_value,
        "l1_ratio": np.nan,
        "params": {"penalty": "l2", "solver": "lbfgs", "C": c_value, "class_weight": "balanced", "max_iter": 5000},
    })

    model_specs.append({
        "model_name": f"l1_C{c_value:g}",
        "model_family": "l1_logistic",
        "penalty": "l1",
        "C": c_value,
        "l1_ratio": np.nan,
        "params": {"penalty": "l1", "solver": "saga", "C": c_value, "class_weight": "balanced", "max_iter": 5000, "random_state": RANDOM_STATE},
    })

    for l1_ratio in L1_RATIOS:
        model_specs.append({
            "model_name": f"elastic_C{c_value:g}_l1r{l1_ratio:g}",
            "model_family": "elastic_net_logistic",
            "penalty": "elasticnet",
            "C": c_value,
            "l1_ratio": l1_ratio,
            "params": {"penalty": "elasticnet", "solver": "saga", "C": c_value, "l1_ratio": l1_ratio, "class_weight": "balanced", "max_iter": 5000, "random_state": RANDOM_STATE},
        })

model_grid = pd.DataFrame([{k: v for k, v in spec.items() if k != "params"} for spec in model_specs])
display(model_grid)

,model_name,model_family,penalty,C,l1_ratio
0,l2_C0.1,l2_logistic,l2,0.1,NaN
1,l1_C0.1,l1_logistic,l1,0.1,NaN
2,elastic_C0.1_l1r0.5,elastic_net_logistic,elasticnet,0.1,0.5
3,l2_C1,l2_logistic,l2,1.0,NaN
4,l1_C1,l1_logistic,l1,1.0,NaN
5,elastic_C1_l1r0.5,elastic_net_logistic,elasticnet,1.0,0.5


## 4. Run CPCV

The imputer and scaler are fitted inside each training fold only.

In [27]:
def sharpe_ratio(returns: pd.Series) -> float:
    returns = pd.Series(returns).dropna()
    if len(returns) < 2 or returns.std(ddof=1) == 0:
        return np.nan
    return float(returns.mean() / returns.std(ddof=1) * np.sqrt(len(returns)))


results = []
total_candidates = len(label_files) * len(model_specs)
candidate_number = 0

for _, label_row in label_files.iterrows():
    data, feature_cols, embargo_observations = make_model_data(label_row)
    splits = make_cpcv_splits(data, label_row["instrument"], embargo_observations)

    for spec in model_specs:
        candidate_number += 1
        print(f"{candidate_number}/{total_candidates}: {label_row['instrument']} | {label_row['config_name']} | {spec['model_name']}")

        for split in splits:
            train = data.iloc[split["train_idx"]]
            test = data.iloc[split["test_idx"]]
            y_train = train["metalabel"].astype(int)
            y_test = test["metalabel"].astype(int)

            if y_train.nunique() < 2:
                continue

            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            x_train = imputer.fit_transform(train[feature_cols])
            x_test = imputer.transform(test[feature_cols])
            x_train = scaler.fit_transform(x_train)
            x_test = scaler.transform(x_test)

            model = LogisticRegression(**spec["params"])
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always", ConvergenceWarning)
                warnings.simplefilter("always", RuntimeWarning)
                model.fit(x_train, y_train)
                y_score = model.predict_proba(x_test)[:, 1]

            convergence_warnings = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
            runtime_warnings = sum(issubclass(w.category, RuntimeWarning) for w in caught)

            y_pred = (y_score >= PROBA_THRESHOLD).astype(int)
            predicted_trade_returns = test.loc[y_pred == 1, "signed_touch_return"]

            results.append({
                "instrument": label_row["instrument"],
                "config_name": label_row["config_name"],
                "model_name": spec["model_name"],
                "model_family": spec["model_family"],
                "penalty": spec["penalty"],
                "C": spec["C"],
                "l1_ratio": spec["l1_ratio"],
                "split_id": split["split_id"],
                "train_rows": split["train_rows"],
                "test_rows": split["test_rows"],
                "embargo_observations": embargo_observations,
                "auc": roc_auc_score(y_test, y_score) if y_test.nunique() == 2 else np.nan,
                "brier": brier_score_loss(y_test, y_score),
                "log_loss": log_loss(y_test, y_score, labels=[0, 1]),
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1": f1_score(y_test, y_pred, zero_division=0),
                "trade_count": int((y_pred == 1).sum()),
                "sharpe": sharpe_ratio(predicted_trade_returns),
                "nonzero_coef_count": int((np.abs(model.coef_.ravel()) > 1e-10).sum()),
                "convergence_warning_count": int(convergence_warnings),
                "runtime_warning_count": int(runtime_warnings),
                "feature_count": len(feature_cols),
                "label_rows": len(data),
            })

path_level_results = pd.DataFrame(results)
assert not path_level_results.empty
display(path_level_results.head())

1/6: cl1s | ewma_10d_tp2_sl2 | l2_C0.1
2/6: cl1s | ewma_10d_tp2_sl2 | l1_C0.1
3/6: cl1s | ewma_10d_tp2_sl2 | elastic_C0.1_l1r0.5
4/6: cl1s | ewma_10d_tp2_sl2 | l2_C1
5/6: cl1s | ewma_10d_tp2_sl2 | l1_C1
6/6: cl1s | ewma_10d_tp2_sl2 | elastic_C1_l1r0.5


,instrument,config_name,model_name,model_family,penalty,C,l1_ratio,split_id,train_rows,test_rows,embargo_observations,auc,accuracy,precision,recall,f1,trade_count,sharpe,nonzero_coef_count,convergence_warning_count,runtime_warning_count,feature_count,label_rows
0,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,0,205,108,10,0.775188,0.592593,0.463415,1.000000,0.633333,82,5.111824,143,0,201,144,323
1,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,1,188,108,10,0.688682,0.592593,0.571429,0.912281,0.702703,91,5.410639,143,0,201,144,323
2,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,2,188,108,10,0.636554,0.583333,0.642857,0.590164,0.615385,56,4.471503,143,0,201,144,323
3,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,3,187,108,10,0.694444,0.518519,0.477778,0.895833,0.623188,90,3.508052,143,0,183,144,323
4,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,4,196,107,10,0.670579,0.654206,0.765625,0.690141,0.725926,64,5.034362,143,0,225,144,323


## 5. Summarise and choose candidates

In [28]:
group_cols = ["instrument", "config_name", "model_name", "model_family", "penalty", "C", "l1_ratio"]

candidate_summary = (
    path_level_results
    .groupby(group_cols, dropna=False, as_index=False)
    .agg(
        mean_auc=("auc", "mean"),
        median_auc=("auc", "median"),
        std_auc=("auc", "std"),
        valid_auc_paths=("auc", lambda x: x.notna().sum()),
        mean_brier=("brier", "mean"),
        mean_log_loss=("log_loss", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        median_sharpe=("sharpe", "median"),
        mean_sharpe=("sharpe", "mean"),
        total_trades=("trade_count", "sum"),
        avg_nonzero_coef_count=("nonzero_coef_count", "mean"),
        total_convergence_warnings=("convergence_warning_count", "sum"),
        total_runtime_warnings=("runtime_warning_count", "sum"),
        feature_count=("feature_count", "first"),
        label_rows=("label_rows", "first"),
        embargo_observations=("embargo_observations", "first"),
    )
)

candidate_summary = candidate_summary[
    candidate_summary["valid_auc_paths"] >= 0.8 * candidate_summary["valid_auc_paths"].max()
].copy()

candidate_summary = candidate_summary.sort_values(
    ["instrument", "mean_auc", "std_auc", "median_sharpe", "total_trades"],
    ascending=[True, False, True, False, False],
).reset_index(drop=True)

best_by_instrument = candidate_summary.groupby("instrument", as_index=False, group_keys=False).head(1).reset_index(drop=True)
best_by_instrument_config = candidate_summary.groupby(["instrument", "config_name"], as_index=False, group_keys=False).head(1).reset_index(drop=True)

display(candidate_summary.head(30))
display(best_by_instrument)

,instrument,config_name,model_name,model_family,penalty,C,l1_ratio,mean_auc,median_auc,valid_auc_paths,mean_accuracy,mean_precision,mean_recall,mean_f1,median_sharpe,mean_sharpe,total_trades,avg_nonzero_coef_count,total_convergence_warnings,total_runtime_warnings,feature_count,label_rows,embargo_observations
0,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,0.629832,0.638889,15,0.566326,0.413049,0.578147,0.451524,2.011836,2.096052,813,143.000000,0,2985,144,323,10
1,cl1s,ewma_10d_tp2_sl2,elastic_C0.1_l1r0.5,elastic_net_logistic,elasticnet,0.1,0.5,0.621806,0.608904,15,0.571986,0.431618,0.633813,0.486480,2.781410,2.848288,870,25.600000,0,45,144,323,10
2,cl1s,ewma_10d_tp2_sl2,elastic_C1_l1r0.5,elastic_net_logistic,elasticnet,1.0,0.5,0.617865,0.621556,15,0.561307,0.395562,0.597229,0.447312,2.614659,2.201184,837,71.400000,0,45,144,323,10
3,cl1s,ewma_10d_tp2_sl2,l2_C1,l2_logistic,l2,1.0,NaN,0.616874,0.642483,15,0.544594,0.386777,0.587959,0.432861,2.432378,2.133142,844,143.000000,0,5829,144,323,10
4,cl1s,ewma_10d_tp2_sl2,l1_C1,l1_logistic,l1,1.0,NaN,0.605318,0.607692,15,0.551390,0.389490,0.576510,0.436216,2.356901,2.107815,823,41.066667,0,45,144,323,10
5,cl1s,ewma_10d_tp2_sl2,l1_C0.1,l1_logistic,l1,0.1,NaN,0.598013,0.593074,15,0.565853,0.434952,0.625083,0.471486,2.508791,2.776939,874,9.133333,0,45,144,323,10


,instrument,config_name,model_name,model_family,penalty,C,l1_ratio,mean_auc,median_auc,valid_auc_paths,mean_accuracy,mean_precision,mean_recall,mean_f1,median_sharpe,mean_sharpe,total_trades,avg_nonzero_coef_count,total_convergence_warnings,total_runtime_warnings,feature_count,label_rows,embargo_observations
0,cl1s,ewma_10d_tp2_sl2,l2_C0.1,l2_logistic,l2,0.1,NaN,0.629832,0.638889,15,0.566326,0.413049,0.578147,0.451524,2.011836,2.096052,813,143.0,0,2985,144,323,10


## 6. Fit final selected models and inspect coefficients

In [29]:
final_models = {}
coefficient_tables = {}

for _, selected in best_by_instrument.iterrows():
    label_row = label_files[
        (label_files["instrument"] == selected["instrument"])
        & (label_files["config_name"] == selected["config_name"])
    ].iloc[0]
    spec = next(s for s in model_specs if s["model_name"] == selected["model_name"])

    data, feature_cols, embargo_observations = make_model_data(label_row)
    y = data["metalabel"].astype(int)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x = imputer.fit_transform(data[feature_cols])
    x = scaler.fit_transform(x)

    model = LogisticRegression(**spec["params"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        warnings.simplefilter("always", RuntimeWarning)
        model.fit(x, y)

    coefficients = pd.DataFrame({"feature": feature_cols, "coefficient": model.coef_.ravel()})
    coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
    coefficients = coefficients.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

    final_models[selected["instrument"]] = {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "selected_config": selected.to_dict(),
    }
    coefficient_tables[selected["instrument"]] = coefficients

    print(f"{selected['instrument']} | {selected['config_name']} | {selected['model_name']}")
    print("Rows:", len(data), "Features:", len(feature_cols), "Embargo:", embargo_observations)
    print("Final fit warnings:", len(caught))
    print("Top positive coefficients")
    display(coefficients.sort_values("coefficient", ascending=False).head(15))
    print("Top negative coefficients")
    display(coefficients.sort_values("coefficient", ascending=True).head(15))

cl1s | ewma_10d_tp2_sl2 | l2_C0.1
Rows: 323 Features: 144 Embargo: 10
Final fit warnings: 216
Top positive coefficients


,feature,coefficient,abs_coefficient
0,vol_ratio_63_126d,0.680597,0.680597
1,mom_sign_252d,0.526672,0.526672
2,mom_sign_20d,0.413900,0.413900
5,bb_width_zscore,0.380522,0.380522
9,macd_zscore,0.314318,0.314318
12,hmm_market_calm_positive,0.284626,0.284626
13,donchian_pos_252d,0.284289,0.284289
14,donchian_pos_20d,0.281678,0.281678
15,macd_hist,0.267726,0.267726
17,ret_spread_5_20,0.257406,0.257406


Top negative coefficients


,feature,coefficient,abs_coefficient
3,uo,-0.399786,0.399786
4,mom_sign_63d,-0.399305,0.399305
6,sma50_slope,-0.334394,0.334394
7,ret_63d,-0.329249,0.329249
8,zscore_63d,-0.317798,0.317798
10,ret_126d,-0.308939,0.308939
11,signal_x_hmm_confidence,-0.284777,0.284777
16,vol_126d,-0.263065,0.263065
18,rsi_14d,-0.256353,0.256353
19,rsi_21d,-0.256126,0.256126


## 7. Optional save

In [ ]:
if SAVE_OUTPUTS:
    import joblib

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path_level_results.to_csv(OUTPUT_DIR / "path_level_results.csv", index=False)
    candidate_summary.to_csv(OUTPUT_DIR / "candidate_summary.csv", index=False)
    best_by_instrument.to_csv(OUTPUT_DIR / "selected_configs.csv", index=False)
    best_by_instrument_config.to_csv(OUTPUT_DIR / "best_by_instrument_config.csv", index=False)

    for instrument, model_bundle in final_models.items():
        joblib.dump(model_bundle, OUTPUT_DIR / f"{instrument}_final_logistic_model.joblib")
        coefficient_tables[instrument].to_csv(OUTPUT_DIR / f"{instrument}_coefficients.csv", index=False)

    print("Saved outputs to", OUTPUT_DIR)
else:
    print("SAVE_OUTPUTS is False, so nothing was written.")